# Joint disease-conditioned GCN prioritization

This notebook trains one shared GCN encoder across all diseases. Each training sample supplies the same PPI graph, a disease-specific seed indicator, and a disease ID. A learned disease embedding is combined with every gene representation before producing one score per gene.

Known genes are split into outer training and held-out test sets. Only outer-training genes can become visible seeds or positive labels; held-out genes are also excluded from sampled negatives.

In [18]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'bioGraph').is_dir():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.gcn_prioritization import predict_from_seed_genes, train_all_diseases

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load the graph and disease associations

The processed subgraph keeps this example practical on a laptop. Replace `ppi_path` with `data/raw/PPI202207.txt` to train on the complete PPI network.

In [19]:
ppi_path = project_root / 'data' / 'processed' / 'subgraph_5377.txt'
disease_path = project_root / 'data' / 'raw' / 'pcbi.1004120.s004.txt'

graph = load_ppi_graph(ppi_path)
diseases = load_disease_genes(disease_path)

print(f'Graph: {graph.number_of_nodes():,} genes, {graph.number_of_edges():,} interactions')
print(f'Diseases: {len(diseases)}')

Graph: 5,377 genes, 94,987 interactions
Diseases: 70


## Train one model jointly across all diseases

The returned model contains one shared encoder and one disease-embedding table. There is no encoder pretraining and no per-disease fine-tuning. `keep_details=True` retains rankings and score tensors for inspection below.

In [26]:
result = train_all_diseases(
    graph,
    diseases,
    k_values=(25, 300),
    hidden_dim=32*4,
    disease_embedding_dim=16*2,
    epochs=50,
    learning_rate=0.01,
    weight_decay=1e-4,
    negative_ratio=5,
    train_fraction=0.75,
    inner_seed_fraction=2/3,
    seed=0,
    task_batch_size=16,
    keep_details=True,
)

model = result['model']
disease_results = result['disease_results']
print(f"Finished {len(result['losses'])} epochs on {result['device']}")
print(f"Final pairwise loss: {result['losses'][-1]:.4f}")

Epoch   1/50: pairwise loss=0.6947
Epoch   2/50: pairwise loss=0.7046
Epoch   3/50: pairwise loss=0.6985
Epoch   4/50: pairwise loss=0.6891
Epoch   5/50: pairwise loss=0.6913
Epoch   6/50: pairwise loss=0.6880
Epoch   7/50: pairwise loss=0.7013
Epoch   8/50: pairwise loss=0.6813
Epoch   9/50: pairwise loss=0.6876
Epoch  10/50: pairwise loss=0.6892
Epoch  11/50: pairwise loss=0.6719
Epoch  12/50: pairwise loss=0.6874
Epoch  13/50: pairwise loss=0.6706
Epoch  14/50: pairwise loss=0.6689
Epoch  15/50: pairwise loss=0.6607
Epoch  16/50: pairwise loss=0.6607
Epoch  17/50: pairwise loss=0.6565
Epoch  18/50: pairwise loss=0.6551
Epoch  19/50: pairwise loss=0.6453
Epoch  20/50: pairwise loss=0.6481
Epoch  21/50: pairwise loss=0.6279
Epoch  22/50: pairwise loss=0.6185
Epoch  23/50: pairwise loss=0.6271
Epoch  24/50: pairwise loss=0.6243
Epoch  25/50: pairwise loss=0.5884
Epoch  26/50: pairwise loss=0.6122
Epoch  27/50: pairwise loss=0.6287
Epoch  28/50: pairwise loss=0.6216
Epoch  29/50: pairwi

## Inspect held-out performance

Each row is computed against that disease's held-out test genes. Training genes are excluded from its final candidate ranking.

In [27]:
metric_rows = [
    {'disease': name, **details['metrics']}
    for name, details in disease_results.items()
]
metrics_by_disease = pd.DataFrame(metric_rows).set_index('disease')
display(metrics_by_disease)
display(metrics_by_disease.mean().rename('mean across diseases'))

,recall@25,ap@25,recall@300,ap@300
disease,,,,
adrenal gland diseases,0.250000,0.020833,0.250000,0.020833
alzheimer disease,0.142857,0.047619,0.285714,0.050980
amino acid metabolism inborn errors,0.307692,0.111661,0.692308,0.154206
amyotrophic lateral sclerosis,0.000000,0.000000,0.200000,0.006897
anemia aplastic,0.600000,0.231579,1.000000,0.271500
...,...,...,...,...
spondylarthropathies,0.250000,0.062500,0.250000,0.062500
tauopathies,0.000000,0.000000,0.111111,0.000741
uveal diseases,0.000000,0.000000,0.250000,0.002155


recall@25     0.114379
ap@25         0.044131
recall@300    0.268121
ap@300        0.049117
Name: mean across diseases, dtype: float64

## Inspect one disease and run a conditioned query

Inference must use the disease ID that selects the learned embedding. The query ranking excludes the supplied seed genes.

In [28]:
disease_name = 'breast neoplasms'
details = disease_results[disease_name]
display(pd.Series(details['metrics'], name=disease_name))
display(pd.DataFrame(details['ranking'][:10]))

query_seed_genes = details['train_genes'][:10]
query_ranking = predict_from_seed_genes(
    model,
    result['graph_data'],
    query_seed_genes,
    disease_id=result['disease_to_id'][disease_name],
)
pd.DataFrame(query_ranking[:20])

recall@25     0.000000
ap@25         0.000000
recall@300    0.200000
ap@300        0.002099
Name: breast neoplasms, dtype: float64

,gene_id,symbol,score
0,162998,OR7D2,6.955850
1,6714,SRC,5.069878
2,1457,CSNK2A1,5.023652
3,5566,PRKACA,4.992257
4,7157,TP53,4.816105
5,5578,PRKCA,4.582545
6,5594,MAPK1,4.384056
7,5595,MAPK3,4.292998
8,1017,CDK2,4.104545
9,983,CDK1,4.103580


,gene_id,symbol,score
0,6714,SRC,4.991540
1,5566,PRKACA,4.940485
2,1457,CSNK2A1,4.827580
3,7157,TP53,4.742357
4,5578,PRKCA,4.532338
5,5594,MAPK1,4.320224
6,5595,MAPK3,4.219779
7,1017,CDK2,4.045974
8,983,CDK1,4.012686
9,2932,GSK3B,3.949370
